# Лабораторная работа 1

## Введение и воспроизводимость исследований

**Вариант 14:** Employee Attrition (IBM HR).

**Задача:** классификация.

**Целевая переменная:** `Attrition`.

## 1. Импорт библиотек

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Настройки проекта

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "employee_attrition_with_missing.csv"
TARGET_COLUMN = "Attrition"
RANDOM_STATE = 42
MISSING_COLUMNS = ["MonthlyIncome", "TotalWorkingYears"]
MISSING_FRACTION = 0.10

RAW_DATA_PATH

## 3. Загрузка данных

In [ ]:
raw_data = pd.read_csv(RAW_DATA_PATH)

raw_data.shape

In [ ]:
raw_data.head()

Список признаков датасета.

In [ ]:
raw_data.columns.tolist()

## 4. Описание признаков

| Колонка | Описание |
|---|---|
| `Age` | Возраст сотрудника. |
| `Attrition` | Целевая переменная: ушел сотрудник из компании или нет. |
| `BusinessTravel` | Частота рабочих командировок. |
| `DailyRate` | Дневная ставка оплаты. |
| `Department` | Подразделение сотрудника. |
| `DistanceFromHome` | Расстояние от дома до работы. |
| `Education` | Уровень образования. |
| `EducationField` | Область образования. |
| `EmployeeCount` | Технический признак с количеством сотрудников в записи. |
| `EmployeeNumber` | Уникальный номер сотрудника. |
| `EnvironmentSatisfaction` | Оценка удовлетворенности рабочей средой. |
| `Gender` | Пол сотрудника. |
| `HourlyRate` | Почасовая ставка оплаты. |
| `JobInvolvement` | Оценка вовлеченности в работу. |
| `JobLevel` | Уровень должности. |
| `JobRole` | Роль или должность сотрудника. |
| `JobSatisfaction` | Оценка удовлетворенности работой. |
| `MaritalStatus` | Семейное положение. |
| `MonthlyIncome` | Ежемесячный доход. |
| `MonthlyRate` | Месячная ставка оплаты. |
| `NumCompaniesWorked` | Количество компаний, в которых сотрудник работал ранее. |
| `Over18` | Признак совершеннолетия. |
| `OverTime` | Наличие сверхурочной работы. |
| `PercentSalaryHike` | Процент повышения зарплаты. |
| `PerformanceRating` | Оценка эффективности работы. |
| `RelationshipSatisfaction` | Оценка удовлетворенности отношениями на работе. |
| `StandardHours` | Стандартное количество рабочих часов. |
| `StockOptionLevel` | Уровень участия в программе опционов компании. |
| `TotalWorkingYears` | Общий трудовой стаж. |
| `TrainingTimesLastYear` | Количество обучений за последний год. |
| `WorkLifeBalance` | Оценка баланса между работой и личной жизнью. |
| `YearsAtCompany` | Количество лет работы в текущей компании. |
| `YearsInCurrentRole` | Количество лет в текущей роли. |
| `YearsSinceLastPromotion` | Количество лет с момента последнего повышения. |
| `YearsWithCurrManager` | Количество лет работы с текущим руководителем. |

## 5. Создание пропусков

В исходном датасете отсутствуют пропущенные значения, поэтому в две числовые колонки внесены искусственные пропуски.

In [ ]:
raw_data.isna().sum().sum()

In [ ]:
data_with_missing = raw_data.copy()

for index, column in enumerate(MISSING_COLUMNS):
    rows = data_with_missing.sample(
        frac=MISSING_FRACTION,
        random_state=RANDOM_STATE + index,
    ).index
    data_with_missing.loc[rows, column] = pd.NA

data_with_missing.to_csv(PROCESSED_DATA_PATH, index=False)
data = pd.read_csv(PROCESSED_DATA_PATH)

data[MISSING_COLUMNS].isna().sum()

In [ ]:
data.shape

## 6. Проверка целевой переменной

In [ ]:
data[TARGET_COLUMN].value_counts()

In [ ]:
data[TARGET_COLUMN].value_counts(normalize=True).round(3)

## 7. Разделение на признаки и целевую переменную

In [ ]:
X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN]

X.shape, y.shape

## 8. Разделение на обучающую и тестовую выборки

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

In [ ]:
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

## 9. Определение числовых и категориальных признаков

In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

len(numeric_columns), len(categorical_columns)

In [ ]:
numeric_columns

In [ ]:
categorical_columns

## 10. Предобработка признаков

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

preprocessor

## 11. Проверка предобработки

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

X_train_prepared.shape, X_test_prepared.shape

Размерность после предобработки.